# 시계열 예측 — DeterministicProcess + LinearRegression (Kaggle Time Series 강좌) — Colab

Kaggle [**Time Series 강좌**](https://www.kaggle.com/learn/time-series) 의 핵심 기법을, 그 강좌의 캡스톤 대회 [**Store Sales — Time Series Forecasting**](https://www.kaggle.com/c/store-sales-time-series-forecasting) 에 적용해 제출합니다.

**강좌 레슨 → 코드 매핑**

| Time Series 강좌 | 이 노트북 |
|---|---|
| Linear Regression with Time Series | 시간 특징 `X` 로 `LinearRegression` |
| Trend | `DeterministicProcess(order=1)` 선형 추세 |
| Seasonality | 주간 계절성(`seasonal=True`) + 연간 `CalendarFourier` |
| Forecasting | `out_of_sample(16)` 으로 미래 16일 예측 |

54개 매장 × 33개 상품군 = **1782개 시계열을 wide 행렬로 한 번에** 회귀합니다(강좌 *Seasonality* 레슨 방식).

> ⚙️ statsmodels + scikit-learn (CPU, 수십 초). train.csv ~120MB 다운로드.
> 🔑 **첫 실행 시 Rules 수락**: 403 이면 [대회 규칙](https://www.kaggle.com/c/store-sales-time-series-forecasting/rules)에서 'I Understand and Accept' 1회.
> ⚠️ 노트북에 API 토큰이 평문으로 들어 있습니다. 외부 공유/업로드 금지.

## 0. 설치 · 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "statsmodels", "scikit-learn", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("준비 완료")

## 1. Kaggle 에서 Store Sales 데이터 다운로드
403 이면 대회 Rules 를 한 번 수락(위 안내).

In [ ]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("store-sales-time-series-forecasting", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:160])
    print("→ 403 이면 https://www.kaggle.com/c/store-sales-time-series-forecasting/rules 에서 'I Understand and Accept' 1회 후 재실행")

## 2. 데이터 준비 · DeterministicProcess + LinearRegression & 제출 파일 생성
`train`/`test` 로드 → 매출을 **wide 행렬**(행=날짜, 열=(store_nbr, family))로, 2017년만 사용. **`DeterministicProcess`**(상수 + 선형 추세 `order=1` + 주간 계절성 + 연간 `CalendarFourier`)로 시간 특징 `X` 를 만들고 **`LinearRegression`** 으로 1782개 시계열을 한 번에 학습 → **`out_of_sample(16)`** 으로 2017-08-16~31 예측(음수는 0 클립) → `test` 의 `id` 에 매핑해 `submission.csv`(`id, sales`) 저장.

In [ ]:
##########################
# Your code here
##########################

## 3. 제출 파일 내려받기

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Store Sales 제출 페이지](https://www.kaggle.com/c/store-sales-time-series-forecasting/submit) 에 업로드.

Kaggle [Time Series 강좌](https://www.kaggle.com/learn/time-series)의 핵심 — **DeterministicProcess(추세+계절성+CalendarFourier) + LinearRegression** 으로 1782개 시계열을 한 번에 예측. 더: 계절성 차수↑, oil/holiday 외생변수, 지연(lag) 특징, 하이브리드(선형+부스팅 잔차).